# 论文标题相似度去重 Demo

> 基于标题相似度将重复文献分组，选择最权威版本作为主记录（不保证能完整聚合 preprint 与正式版）

**场景**: 科研 Agent 在检索时常遇到同一篇论文的 preprint（arXiv）、正式发表版（Nature/Science）、以及 PDF/Web 多来源副本。需要将它们聚合为一条 canonical 记录，避免重复引用。

**使用接口**: agentic-search, meta-search, content

**预估调用量**: ~10–20 次 API 调用 / 一次去重任务

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
!pip install httpx
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 语义检索候选文献

用 agentic-search 获取与某主题相关的所有片段


In [ ]:
import os
import asyncio
import httpx
from collections import defaultdict

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def search_candidates(query: str, top_k: int = 50):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search", headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        return resp.json()["hits"]

hits = asyncio.run(search_candidates("Attention Is All You Need transformer"))
print(f"Raw hits: {len(hits)}")


## Step 3: 按标题相似度聚合

将同一篇论文的不同版本分组


In [ ]:
from difflib import SequenceMatcher

def title_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def cluster_by_title(hits, threshold=0.85):
    clusters = []
    used = set()
    for i, h in enumerate(hits):
        if i in used:
            continue
        group = [h]
        used.add(i)
        for j in range(i + 1, len(hits)):
            if j in used:
                continue
            if title_similarity(h["title"], hits[j]["title"]) >= threshold:
                group.append(hits[j])
                used.add(j)
        clusters.append(group)
    return clusters

clusters = cluster_by_title(hits)
print(f"Clustered into {len(clusters)} unique papers")
for c in clusters[:3]:
    print(f"  [{len(c)} versions] {c[0]['title'][:60]}")


## Step 4: 确认正式版本并生成 canonical pack

用 meta-search 查询 DOI 信息，选择最权威版本


In [ ]:
async def find_primary(cluster):
    """\u4ece\u4e00\u7ec4\u7248\u672c\u4e2d\u627e\u5230\u6700\u6743\u5a01\u7684\u6b63\u5f0f\u53d1\u8868\u7248"""
    # \u4f18\u5148\u7ea7: \u6709 DOI > \u6709 venue > arXiv
    best = cluster[0]
    for item in cluster:
        # \u7528 meta-search \u67e5\u8be2\u66f4\u591a\u5143\u6570\u636e
        async with httpx.AsyncClient(timeout=30) as client:
            resp = await client.post(
                f"{BASE}/meta-search", headers=HEADERS,
                json={
                    "query": item["title"],
                    "filters": [],
                    "page": 1, "page_size": 1
                }
            )
            if resp.status_code == 200:
                results = resp.json().get("results", [])
                if results and results[0].get("doi"):
                    best = item
                    break
    return {
        "title": best["title"],
        "primary_doc_id": best["doc_id"],
        "versions": [{"doc_id": v["doc_id"], "title": v["title"]} for v in cluster]
    }

async def build_canonical_pack(clusters):
    pack = []
    for cluster in clusters[:10]:
        canonical = await find_primary(cluster)
        pack.append(canonical)
    return pack

pack = asyncio.run(build_canonical_pack(clusters))
print(f"Canonical pack: {len(pack)} unique papers")


## 注意事项

- 标题相似度阈值 0.85 适合大多数场景，可根据领域调整
- 对于有 DOI 的论文，可直接用 DOI 做精确去重
- 建议保留所有版本的 doc_id，以便后续读取不同版本的全文
- canonical pack 可作为下游 RAG/Agent 的标准输入


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
